# 📖 Notebook 3: Deduplication & Fraud Detection

Clicks mean money. Every fake or duplicate click that sneaks through costs advertisers real dollars. In this notebook we build three layers of defense:

1. **Impression IDs** — a unique token per ad view so we can tell duplicates apart
2. **HMAC Signing** — cryptographic proof the impression ID is genuine (not fabricated)
3. **Redis Dedup Cache** — fast O(1) lookup to reject duplicates *before* they enter Kafka

```
User clicks ad
      │
      ▼
┌──────────────┐  1. Verify HMAC   ┌──────────────┐
│ Click        │─────────────────▶│   HMAC       │  Invalid? → Reject ❌
│ Processor    │                   │   Check      │
└──────┬───────┘                   └──────────────┘
       │  2. Check Redis
       ▼
┌──────────────┐  Seen before? → Reject ❌
│    Redis     │
│  Dedup Cache │  New? → Mark as seen ✅
└──────┬───────┘
       │  3. Forward to Kafka
       ▼
┌──────────────┐
│    Kafka     │  Only verified, unique clicks make it here
└──────────────┘
```

## Learning Objectives

By the end of this notebook you will understand:
- What an impression ID is and why it's better than user-based dedup
- How HMAC signing prevents fabricated impression IDs
- How to use Redis as a fast dedup cache with TTL (time-to-live)
- How to put the full pipeline together: verify → dedup → produce → aggregate
- Basic anomaly detection patterns for click fraud

## 🛠️ Setup

Make sure the infrastructure is running:

```bash
cd system-designs/ad-click-aggregator
docker-compose up -d
```

### Visualization Tools

- **RedisInsight** (Redis GUI): http://localhost:5540  
  Watch deduplication keys appear in real-time!
- **Kafka UI**: http://localhost:8081
- **Adminer** (PostgreSQL): http://localhost:8080

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import time
import uuid
import hmac
import hashlib
import random
from datetime import datetime, timezone, timedelta
from collections import defaultdict
from confluent_kafka import Producer, Consumer, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic

# ── Connection settings ─────────────────────────────────────
DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "adclick_demo", "user": "demo", "password": "demo"
}
KAFKA_BROKER = "localhost:9094"
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Verify connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Redis")
except Exception as e:
    print(f"❌ Redis: {e}")

try:
    admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
    admin.list_topics(timeout=5)
    print("✅ Kafka")
except Exception as e:
    print(f"❌ Kafka: {e}")

## 🤔 Why Not Just Dedup by User ID?

The simplest dedup idea: "if user X already clicked ad Y, ignore future clicks."

But this breaks **retargeting** — showing the same ad to the same user multiple times on purpose. We *want* to count a click each time the user sees and clicks the ad again.

The fix: instead of deduping by `(user_id, ad_id)`, we dedup by **impression ID** — a unique token generated each time an ad is shown to a user.

In [ ]:
# Demonstrate the difference

print("🤔 User-based vs Impression-based Dedup")
print("=" * 60)

scenarios = [
    {
        "desc": "User sees Nike ad at 9:00, clicks it",
        "user": "alice", "ad": "Nike", "impression": "imp-001",
        "should_count": True
    },
    {
        "desc": "Same user double-clicks (accidental)",
        "user": "alice", "ad": "Nike", "impression": "imp-001",
        "should_count": False
    },
    {
        "desc": "Same user sees Nike ad AGAIN at 10:00 (retargeted)",
        "user": "alice", "ad": "Nike", "impression": "imp-002",
        "should_count": True
    },
    {
        "desc": "Different user clicks Nike ad",
        "user": "bob", "ad": "Nike", "impression": "imp-003",
        "should_count": True
    },
]

# Track seen impressions
seen_impressions = set()
seen_user_ad = set()

print(f"\n{'#':<3} {'Description':<50} {'User Dedup':<12} {'Imp Dedup':<12} {'Correct?'}")
print("-" * 100)

for i, s in enumerate(scenarios, 1):
    user_ad_key = (s["user"], s["ad"])
    imp_key = s["impression"]

    # User-based dedup: reject if (user, ad) seen before
    user_dedup = "❌ Drop" if user_ad_key in seen_user_ad else "✅ Count"
    seen_user_ad.add(user_ad_key)

    # Impression-based dedup: reject if impression_id seen before
    imp_dedup = "❌ Drop" if imp_key in seen_impressions else "✅ Count"
    seen_impressions.add(imp_key)

    expected = "✅ Count" if s["should_count"] else "❌ Drop"
    user_correct = "✓" if user_dedup == expected else "✗ WRONG"
    imp_correct = "✓" if imp_dedup == expected else "✗ WRONG"

    print(f"{i:<3} {s['desc']:<50} {user_dedup:<12} {imp_dedup:<12} "
          f"User:{user_correct} Imp:{imp_correct}")

print("\n💡 User-based dedup wrongly blocks the retargeted click (#3).")
print("   Impression-based dedup handles all cases correctly.")

## 🔐 Step 1 — HMAC Signing

A malicious user could fabricate impression IDs to generate fake clicks. To prevent this, the **Ad Placement Service** signs each impression ID with a secret key using HMAC-SHA256.

### How HMAC Works (Simply)

1. Server has a **secret key** that only it knows
2. When showing an ad, the server creates: `signature = HMAC(secret, impression_id + ad_id)`
3. The signature is sent to the browser along with the impression ID
4. When a click comes in, the server re-computes the HMAC and checks it matches
5. An attacker can't forge a valid signature because they don't know the secret key

HMAC is extremely fast (microseconds) — just a hash computation, no expensive cryptography.

In [ ]:
# Our secret key (in production, this comes from a secrets manager)
SECRET_KEY = b"super-secret-key-do-not-share"

def sign_impression(impression_id: str, ad_id: int) -> str:
    """
    Create an HMAC signature for an impression.
    The Ad Placement Service calls this when showing an ad.
    """
    message = f"{impression_id}:{ad_id}".encode("utf-8")
    return hmac.new(SECRET_KEY, message, hashlib.sha256).hexdigest()

def verify_impression(impression_id: str, ad_id: int, signature: str) -> bool:
    """
    Verify that an impression signature is valid.
    The Click Processor calls this when a click arrives.
    """
    expected = sign_impression(impression_id, ad_id)
    # Use hmac.compare_digest to prevent timing attacks
    return hmac.compare_digest(expected, signature)

# ── Demo: legitimate impression ──
imp_id = str(uuid.uuid4())
ad_id = 1
sig = sign_impression(imp_id, ad_id)

print("🔐 HMAC Signing Demo")
print("=" * 60)
print(f"\n  Impression ID: {imp_id}")
print(f"  Ad ID:         {ad_id}")
print(f"  Signature:     {sig[:32]}...")
print(f"  Valid?         {verify_impression(imp_id, ad_id, sig)} ✅")

# ── Demo: tampered impression ──
print(f"\n  🚨 Attacker tries to use same impression for a different ad:")
print(f"  Valid for ad 2? {verify_impression(imp_id, 2, sig)} ❌")

# ── Demo: fabricated impression ──
fake_imp = str(uuid.uuid4())
fake_sig = "deadbeef" * 8
print(f"\n  🚨 Attacker fabricates an impression ID:")
print(f"  Valid?         {verify_impression(fake_imp, ad_id, fake_sig)} ❌")

print("\n💡 Without the secret key, attackers can't create valid signatures.")

## ⚡ Step 2 — Redis Dedup Cache

After verifying the HMAC, we need to check if this impression has already been clicked. We use **Redis** because:

- **O(1) lookups** — checking if a key exists takes constant time
- **TTL (Time-To-Live)** — keys auto-expire after 24 hours, so we don't need manual cleanup
- **Tiny memory footprint** — 100M impression IDs × 36 bytes each ≈ 3.6 GB (fits in RAM easily)

The pattern:
1. `SET impression_id 1 NX EX 86400` — set the key **only if it doesn't exist** (NX), expire in 24h (EX)
2. If SET returns True → first click → proceed
3. If SET returns None → duplicate → reject

In [ ]:
r = get_redis()

DEDUP_TTL = 86400  # 24 hours in seconds

def check_and_mark(impression_id: str) -> bool:
    """
    Check if an impression has already been clicked.
    Returns True if this is a NEW click (not a duplicate).
    Returns False if this is a DUPLICATE.
    """
    key = f"dedup:{impression_id}"
    # SET NX = Set if Not eXists. Returns True if set, None if already existed.
    result = r.set(key, "1", nx=True, ex=DEDUP_TTL)
    return result is not None  # True = new, False = duplicate

# Demo
print("⚡ Redis Dedup Cache Demo")
print("=" * 50)

test_imp = str(uuid.uuid4())

result1 = check_and_mark(test_imp)
print(f"\n  First click  (impression {test_imp[:8]}...): "
      f"{'✅ New — proceed' if result1 else '❌ Duplicate — reject'}")

result2 = check_and_mark(test_imp)
print(f"  Second click (impression {test_imp[:8]}...): "
      f"{'✅ New — proceed' if result2 else '❌ Duplicate — reject'}")

result3 = check_and_mark(test_imp)
print(f"  Third click  (impression {test_imp[:8]}...): "
      f"{'✅ New — proceed' if result3 else '❌ Duplicate — reject'}")

# Check the TTL
ttl = r.ttl(f"dedup:{test_imp}")
print(f"\n  ⏰ Key expires in {ttl} seconds ({ttl // 3600} hours)")
print("     After expiry, the same impression ID could be clicked again.")
print("     But by then, the ad impression is long gone from the user's screen.")

### 🔍 Go look in RedisInsight!

Open http://localhost:5540 → Browse keys → search for `dedup:*`.  
You should see the key we just created with its TTL.

## 🛡️ Step 3 — Complete Click Processor

Let's combine everything into a complete click processor that:
1. Verifies the HMAC signature
2. Checks the Redis dedup cache
3. Produces valid clicks to Kafka

In [ ]:
topic_name = "ad-clicks-dedup"

# Create topic
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
existing = admin.list_topics(timeout=10).topics
if topic_name not in existing:
    futures = admin.create_topics([NewTopic(topic_name, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()
    print(f"✅ Created topic '{topic_name}'")
else:
    print(f"ℹ️  Topic '{topic_name}' already exists")

producer = Producer({"bootstrap.servers": KAFKA_BROKER})
r = get_redis()

# Stats
stats = {"accepted": 0, "dup_rejected": 0, "hmac_rejected": 0}

def process_click(impression_id: str, ad_id: int, signature: str,
                  user_id: str, ip_address: str, user_agent: str,
                  event_time: str) -> str:
    """
    Process a single click event through the full validation pipeline.
    Returns: 'accepted', 'hmac_rejected', or 'dup_rejected'
    """
    # Step 1: Verify HMAC
    if not verify_impression(impression_id, ad_id, signature):
        stats["hmac_rejected"] += 1
        return "hmac_rejected"

    # Step 2: Check dedup cache
    if not check_and_mark(impression_id):
        stats["dup_rejected"] += 1
        return "dup_rejected"

    # Step 3: Forward to Kafka (only verified, unique clicks get here)
    event = {
        "ad_id": ad_id,
        "impression_id": impression_id,
        "user_id": user_id,
        "ip_address": ip_address,
        "user_agent": user_agent,
        "event_time": event_time
    }
    producer.produce(topic_name, key=str(ad_id), value=json.dumps(event))
    stats["accepted"] += 1
    return "accepted"

print("🛡️ Click Processor ready")
print("   Pipeline: Verify HMAC → Check Redis Dedup → Produce to Kafka")

## 🧪 Step 4 — Simulate Real Traffic

Let's simulate a realistic mix of:
- ✅ **Legitimate clicks** (valid HMAC, unique impression)
- ❌ **Duplicate clicks** (user double-clicks or refreshes)
- 🚨 **Fabricated clicks** (attacker creates fake impression IDs)

In [ ]:
# Reset stats and dedup cache
stats = {"accepted": 0, "dup_rejected": 0, "hmac_rejected": 0}
# Clean dedup keys from previous runs
for key in r.scan_iter(match="dedup:*", count=1000):
    r.delete(key)

print("🧪 Simulating Mixed Traffic")
print("=" * 60)

# Generate 50 legitimate impressions (pre-signed by Ad Placement Service)
legit_impressions = []
for _ in range(50):
    imp_id = str(uuid.uuid4())
    ad_id = random.randint(1, 10)
    sig = sign_impression(imp_id, ad_id)
    legit_impressions.append((imp_id, ad_id, sig))

now = datetime.now(timezone.utc)

# ── Legitimate clicks (50 unique) ──
print("\n📤 Sending 50 legitimate clicks...")
for imp_id, ad_id, sig in legit_impressions:
    process_click(
        impression_id=imp_id, ad_id=ad_id, signature=sig,
        user_id=f"user_{random.randint(1,200)}",
        ip_address=f"10.0.{random.randint(1,254)}.{random.randint(1,254)}",
        user_agent="Mozilla/5.0",
        event_time=(now + timedelta(seconds=random.randint(0, 60))).isoformat()
    )

# ── Duplicate clicks (replay 15 of the same impressions) ──
print("📤 Sending 15 duplicate clicks (same impressions again)...")
for imp_id, ad_id, sig in random.sample(legit_impressions, 15):
    process_click(
        impression_id=imp_id, ad_id=ad_id, signature=sig,
        user_id=f"user_{random.randint(1,200)}",
        ip_address=f"10.0.{random.randint(1,254)}.{random.randint(1,254)}",
        user_agent="Mozilla/5.0",
        event_time=now.isoformat()
    )

# ── Fabricated clicks (fake impression IDs with bad signatures) ──
print("📤 Sending 10 fabricated clicks (fake signatures)...")
for _ in range(10):
    process_click(
        impression_id=str(uuid.uuid4()),
        ad_id=random.randint(1, 10),
        signature="fake_signature_" + str(random.randint(1, 1000)),
        user_id=f"bot_{random.randint(1, 50)}",
        ip_address="1.2.3.4",
        user_agent="BotBrowser/1.0",
        event_time=now.isoformat()
    )

producer.flush(timeout=10)

# Results
total = sum(stats.values())
print(f"\n📊 Results:")
print(f"   Total clicks received: {total}")
print(f"   ✅ Accepted (valid & unique):   {stats['accepted']:>3}  ({stats['accepted']/total*100:.0f}%)")
print(f"   ❌ Rejected (duplicate):         {stats['dup_rejected']:>3}  ({stats['dup_rejected']/total*100:.0f}%)")
print(f"   🚨 Rejected (invalid HMAC):      {stats['hmac_rejected']:>3}  ({stats['hmac_rejected']/total*100:.0f}%)")
print()
print(f"   Only {stats['accepted']} of {total} clicks made it to Kafka.")
print(f"   Advertisers are protected from paying for {stats['dup_rejected'] + stats['hmac_rejected']} fake/duplicate clicks!")

## 🕵️ Step 5 — Simple Anomaly Detection

Beyond dedup, we can flag suspicious patterns. Here are some simple heuristics:

| Signal | What It Catches |
|--------|----------------|
| Too many clicks from one IP in a short window | Click farms |
| Click rate spike for one ad | Bot attack on a specific ad |
| Clicks without corresponding impressions | Fabricated traffic |

Let's implement a basic IP-based rate limiter using Redis.

In [ ]:
# IP-based rate limiting with Redis sliding window

MAX_CLICKS_PER_IP_PER_MINUTE = 5

def check_ip_rate(ip_address: str) -> dict:
    """
    Check if an IP has exceeded the click rate limit.
    Uses a Redis key with a 60-second TTL.

    Returns: {"allowed": bool, "current_count": int, "limit": int}
    """
    key = f"rate:{ip_address}"

    # INCR atomically increments (and creates the key if it doesn't exist)
    count = r.incr(key)

    # Set TTL only on first increment (when count == 1)
    if count == 1:
        r.expire(key, 60)

    return {
        "allowed": count <= MAX_CLICKS_PER_IP_PER_MINUTE,
        "current_count": count,
        "limit": MAX_CLICKS_PER_IP_PER_MINUTE
    }

# Demo: simulate a click farm
print("🕵️ IP Rate Limiting Demo")
print("=" * 50)
print(f"   Limit: {MAX_CLICKS_PER_IP_PER_MINUTE} clicks per IP per minute\n")

# Clean up previous rate limit keys
for key in r.scan_iter(match="rate:*", count=1000):
    r.delete(key)

farm_ip = "203.0.113.42"
normal_ip = "198.51.100.7"

print(f"Suspicious IP ({farm_ip}):")
for i in range(8):
    result = check_ip_rate(farm_ip)
    status = "✅ Allowed" if result["allowed"] else "🚨 BLOCKED"
    print(f"   Click {i+1}: {status} (count: {result['current_count']}/{result['limit']})")

print(f"\nNormal IP ({normal_ip}):")
for i in range(3):
    result = check_ip_rate(normal_ip)
    status = "✅ Allowed" if result["allowed"] else "🚨 BLOCKED"
    print(f"   Click {i+1}: {status} (count: {result['current_count']}/{result['limit']})")

print("\n💡 The rate limiter blocks the suspicious IP after 5 clicks.")
print("   Real fraud detection systems use ML models for more accuracy,")
print("   but simple rate limiting catches the obvious cases.")

## 📊 Step 6 — Memory Budget: Can Redis Handle This?

Let's calculate whether Redis can store all our dedup keys.

In [ ]:
print("📊 Redis Memory Budget")
print("=" * 50)
print()

clicks_per_day = 100_000_000  # 100M clicks/day (from requirements)
key_size_bytes = 36 + 6       # "dedup:" prefix + UUID = ~42 bytes
value_size_bytes = 1           # just "1"
redis_overhead_bytes = 64      # Redis internal overhead per key
bytes_per_key = key_size_bytes + value_size_bytes + redis_overhead_bytes

total_bytes = clicks_per_day * bytes_per_key
total_gb = total_bytes / (1024 ** 3)

print(f"  Clicks per day:          {clicks_per_day:>15,}")
print(f"  Bytes per key (w/ overhead): {bytes_per_key:>12} bytes")
print(f"  Total memory needed:     {total_gb:>14.1f} GB")
print()

redis_instance_ram = 64  # typical Redis instance
print(f"  Typical Redis RAM:       {redis_instance_ram:>14} GB")
print(f"  Keys fit in one instance? {'✅ Yes!' if total_gb < redis_instance_ram else '❌ Need cluster'}")
print()
print("💡 ~10 GB for 100M dedup keys is very manageable.")
print("   Keys auto-expire after 24h via TTL, so memory stays bounded.")
print("   For higher scale, Redis Cluster shards across multiple nodes.")

## 🔄 Step 7 — Full Pipeline End-to-End

Let's run the complete pipeline: Ad Placement generates signed impressions → Click Processor validates and deduplicates → Kafka → Aggregation Consumer → PostgreSQL.

In [ ]:
# Helper from Notebook 2: window assignment and aggregation
def get_window_start(event_time_str: str, window_seconds: int = 60) -> datetime:
    event_time = datetime.fromisoformat(event_time_str)
    ts = event_time.timestamp()
    ws = (ts // window_seconds) * window_seconds
    return datetime.fromtimestamp(ws, tz=timezone.utc)

# Clean slate
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
conn.commit()
conn.close()
for key in r.scan_iter(match="dedup:*", count=1000):
    r.delete(key)
for key in r.scan_iter(match="rate:*", count=1000):
    r.delete(key)

# ── Phase 1: Ad Placement Service generates signed impressions ──
print("🎯 Phase 1: Ad Placement Service generates 100 impressions")
impressions = []
for _ in range(100):
    imp_id = str(uuid.uuid4())
    ad_id = random.randint(1, 5)
    sig = sign_impression(imp_id, ad_id)
    impressions.append((imp_id, ad_id, sig))

# ── Phase 2: Simulate clicks (including dupes and fakes) ──
print("🖱️  Phase 2: Processing clicks (100 legit + 20 dupes + 10 fakes)")

pipeline_topic = "ad-clicks-full-pipeline"
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
if pipeline_topic not in admin.list_topics(timeout=10).topics:
    futures = admin.create_topics([NewTopic(pipeline_topic, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()

pipeline_producer = Producer({"bootstrap.servers": KAFKA_BROKER})
pipeline_stats = {"accepted": 0, "dup_rejected": 0, "hmac_rejected": 0}

base_time = datetime(2026, 3, 31, 15, 0, 0, tzinfo=timezone.utc)

def pipeline_process(imp_id, ad_id, sig, user_id, ip, ua, event_time):
    if not verify_impression(imp_id, ad_id, sig):
        pipeline_stats["hmac_rejected"] += 1
        return
    if not check_and_mark(imp_id):
        pipeline_stats["dup_rejected"] += 1
        return
    event = {
        "ad_id": ad_id, "impression_id": imp_id, "user_id": user_id,
        "ip_address": ip, "user_agent": ua, "event_time": event_time
    }
    pipeline_producer.produce(pipeline_topic, key=str(ad_id), value=json.dumps(event))
    pipeline_stats["accepted"] += 1

# Legitimate clicks
for imp_id, ad_id, sig in impressions:
    et = (base_time + timedelta(seconds=random.randint(0, 120))).isoformat()
    pipeline_process(imp_id, ad_id, sig, f"user_{random.randint(1,200)}",
                     f"10.0.{random.randint(1,254)}.{random.randint(1,254)}",
                     "Mozilla/5.0", et)

# Duplicate clicks
for imp_id, ad_id, sig in random.sample(impressions, 20):
    et = (base_time + timedelta(seconds=random.randint(0, 120))).isoformat()
    pipeline_process(imp_id, ad_id, sig, f"user_{random.randint(1,200)}",
                     f"10.0.1.1", "Mozilla/5.0", et)

# Fake clicks
for _ in range(10):
    et = base_time.isoformat()
    pipeline_process(str(uuid.uuid4()), random.randint(1,5), "FAKE_SIG",
                     f"bot_{random.randint(1,10)}", "1.2.3.4", "Bot/1.0", et)

pipeline_producer.flush(timeout=10)

total = sum(pipeline_stats.values())
print(f"   ✅ Accepted: {pipeline_stats['accepted']}  "
      f"❌ Dup: {pipeline_stats['dup_rejected']}  "
      f"🚨 HMAC: {pipeline_stats['hmac_rejected']}")

# ── Phase 3: Aggregate ──
print("\n📊 Phase 3: Aggregating from Kafka")

consumer = Consumer({
    "bootstrap.servers": KAFKA_BROKER,
    "group.id": "full-pipeline-" + str(uuid.uuid4())[:8],
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False
})
consumer.subscribe([pipeline_topic])

windows = defaultdict(lambda: {"click_count": 0, "unique_users": set()})
consumed = 0
start = time.time()

while time.time() - start < 15 and consumed < pipeline_stats["accepted"]:
    msg = consumer.poll(timeout=1.0)
    if msg is None:
        continue
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        break
    event = json.loads(msg.value().decode("utf-8"))
    ws = get_window_start(event["event_time"])
    key = (event["ad_id"], ws.isoformat())
    windows[key]["click_count"] += 1
    windows[key]["unique_users"].add(event.get("user_id", "anon"))
    consumed += 1

consumer.close()
print(f"   Consumed {consumed} events, aggregated into {len(windows)} windows")

# ── Phase 4: Flush to PostgreSQL ──
print("\n💾 Phase 4: Flushing to PostgreSQL")
conn = get_db()
cur = conn.cursor()
for (ad_id, ws_iso), data in windows.items():
    cur.execute("""
        INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (ad_id, window_start) DO UPDATE SET
            click_count = click_aggregates.click_count + EXCLUDED.click_count,
            unique_users = click_aggregates.unique_users + EXCLUDED.unique_users,
            updated_at = CURRENT_TIMESTAMP
    """, (ad_id, ws_iso, data["click_count"], len(data["unique_users"])))
conn.commit()

# ── Phase 5: Advertiser query ──
print("\n⚡ Phase 5: Advertiser Dashboard")
cur.execute("""
    SELECT a.title, SUM(ca.click_count) AS clicks
    FROM click_aggregates ca JOIN ads a ON a.id = ca.ad_id
    GROUP BY a.id, a.title ORDER BY clicks DESC
""")
rows = cur.fetchall()
conn.close()

print(f"\n{'Ad Title':<42} {'Clicks':>8}")
print("-" * 52)
for row in rows:
    print(f"{row[0]:<42} {row[1]:>8}")

print(f"\n✅ Full pipeline complete!")
print(f"   {total} clicks in → {pipeline_stats['accepted']} valid clicks stored")
print(f"   {pipeline_stats['dup_rejected'] + pipeline_stats['hmac_rejected']} fraudulent/duplicate clicks blocked")

## 🧹 Cleanup

In [ ]:
# Clean up PostgreSQL
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
conn.commit()
conn.close()
print("🧹 Cleaned PostgreSQL tables")

# Clean up Redis
r = get_redis()
cleaned = 0
for key in r.scan_iter(match="dedup:*", count=1000):
    r.delete(key)
    cleaned += 1
for key in r.scan_iter(match="rate:*", count=1000):
    r.delete(key)
    cleaned += 1
print(f"🧹 Cleaned {cleaned} Redis keys")

print("\n🎉 All done! You can stop the infrastructure with:")
print("   cd system-designs/ad-click-aggregator")
print("   docker-compose down")

## 📚 Summary

### Key Takeaways

1. **Impression IDs** are better than user IDs for dedup because they support retargeting
2. **HMAC signing** prevents attackers from fabricating impression IDs — verification is microsecond-fast
3. **Redis dedup cache** with `SET NX EX` gives O(1) duplicate detection with auto-expiry
4. **Memory budget** is tiny — 100M keys/day fits in ~10 GB of RAM
5. **Rate limiting** by IP is a simple first line of defense against click farms
6. **Dedup before Kafka** — we reject duplicates *before* they enter the stream to prevent cross-window miscounting

### The Complete Architecture

```
Ad Placement Service
  │  generates impression_id + HMAC signature
  ▼
User clicks ad
  │  sends impression_id + signature
  ▼
Click Processor
  ├─ 1. Verify HMAC → reject fakes
  ├─ 2. Redis dedup → reject duplicates
  ├─ 3. IP rate limit → flag anomalies
  └─ 4. Produce to Kafka
           │
           ▼
     Aggregation Consumer
       ├─ Tumbling 1-min windows (event time)
       └─ UPSERT to click_aggregates
                │
                ▼
          Advertiser Dashboard
            sub-second queries!
```

### Real-World Additions

| Feature | What Production Systems Add |
|---------|---------------------------|
| **ML fraud detection** | Models trained on click patterns, not just rate limits |
| **Reconciliation** | Daily batch job re-aggregates from raw events to catch errors |
| **Hot shard mitigation** | Append random suffix to popular ad partition keys |
| **Flink/Spark Streaming** | Replaces our manual consumer with watermarks and exactly-once |
| **Redis Cluster** | Shard dedup keys across nodes for higher throughput |